# Missão Aurora Siger: Relatório Operacional de Pré-Decolagem
**Disciplina:** Atividade Integradora - Ciência da Computação (FIAP)  
**Responsável:**  José Rodrigues da Silva Junior
**Curso:**  Ciência da Computação (On-Line)
**RM:** rm572300

**Link para o Github:**
https://github.com/jose-rodrigues-da-silva-junior/fiap-1ccoa-fase1-missao-aurora-siger



## 1. Geração do Dataset de Telemetria (Simulação Sintética)

Antes de um astronauta sentar na cabine de um foguete real, ele passa milhares de horas em um Simulador de Voo. Por quê? Porque no simulador o erro é um aprendizado, mas na decolagem real, o erro é uma catástrofe.

Nesta etapa da missão, estamos construindo o **"Motor de Eventos"** desse simulador. O script que desenvolvemos gera 100 cenários de telemetria que imitam exatamente o que os sensores da nave enviariam em tempo real.

Imagine que estamos **"alimentando"** o computador de bordo com situações diversas: em algumas leituras, a pressão está perfeita; em outras, simulamos uma falha crítica de energia ou uma temperatura extrema no casco. Esses dados sintéticos são os nossos **"voos de teste virtuais"**. Eles nos permitem treinar nosso algoritmo de decisão para que ele aprenda a dizer "Abortar" no ambiente seguro do código, antes que tenhamos que tomar essa decisão com combustível real nos tanques.

In [ ]:
import pandas as pd
import random
import os
from datetime import datetime, timedelta

def gerar_telemetria_congruente(n_amostras=100):
    """
    Gera o dataset da Missão Aurora com valores baseados nos Critérios de Aceite:
    - Temp Interna: 18 a 25 | Temp Externa: -100 a 60
    - Integridade: 1 | Energia: > 80%
    - Pressão: 3000 a 4500 | Status: Ativo
    """
    random.seed(42)
    dados = []
    tempo_inicial = datetime.now()

    for i in range(n_amostras):
        timestamp = (tempo_inicial + timedelta(seconds=i)).strftime('%Y-%m-%d %H:%M:%S')

        # Geramos dados majoritariamente dentro da faixa, com leves variações para testes
        dados.append({
            'timestamp': timestamp,
            'leitura_id': i + 1,
            # Critério: 18 a 25 (Geramos com pequena margem de erro para simular alertas)
            'temp_interna': round(random.uniform(17.5, 25.5), 2),

            # Critério: -100 a 60
            'temp_externa': round(random.uniform(-105.0, 65.0), 2),

            # Critério: 1 (90% de chance de estar OK)
            'integridade_estrutural': 1 if random.random() > 0.10 else 0,

            # Critério: > 80% (Geramos de 75 a 100 para ter casos de aborto por energia)
            'nivel_energia': round(random.uniform(75.0, 100.0), 2),

            # Critério: 3000 a 4500 PSI
            'pressao_tanques': round(random.uniform(2800, 4700), 2),

            # Critério: "Ativo" (80% Ativo, 15% Alerta, 5% Crítico)
            'status_modulos': random.choices(['Ativo', 'Alerta', 'Critico'], weights=[80, 15, 5])[0]
        })

    df = pd.DataFrame(dados)

    # Organização de pastas no Colab
    diretorio_data = 'data'
    if not os.path.exists(diretorio_data):
        os.makedirs(diretorio_data)

    caminho_final = os.path.join(diretorio_data, 'telemetria_missao_aurora.csv')
    df.to_csv(caminho_final, index=False)

    print(f"Dataset gerado em: {caminho_final}")
    return df

# Execução
df_missao = gerar_telemetria_congruente()
df_missao.head()

Dataset gerado em: data/telemetria_missao_aurora.csv


,timestamp,leitura_id,temp_interna,temp_externa,integridade_estrutural,nivel_energia,pressao_tanques,status_modulos
0,2026-04-01 02:01:00,1,22.62,-100.75,1,80.58,4199.30,Ativo
1,2026-04-01 02:01:01,2,24.64,-90.22,1,75.74,3215.41,Ativo
2,2026-04-01 02:01:02,3,17.71,-71.20,1,88.62,3218.84,Ativo
3,2026-04-01 02:01:03,4,23.98,-103.90,1,92.45,3446.48,Ativo
4,2026-04-01 02:01:04,5,25.16,-47.78,0,77.42,4410.24,Ativo


## 2. Critérios de Missão: Definição de Faixas de Segurança

Antes de processarmos a telemetria, estabelecemos os limites técnicos que garantem a segurança da tripulação e a integridade da **Missão Aurora Siger**. Qualquer valor fora dessas "Faixas Seguras" resultará em **DECOLAGEM ABORTADA**.

### 2.1 Tabela de Parâmetros Operacionais

| Sensor | Faixa Mínima | Faixa Máxima | Critério de Sucesso |
| :--- | :--- | :--- | :--- |
| **Temperatura Interna** | 18°C | 25°C | Conforto térmico e operação de hardware. |
| **Temperatura Externa** | -100°C | 60°C | Resistência térmica do casco. |
| **Integridade Estrutural** | 1 | 1 | Deve ser binário 1 (Sem falhas detectadas). |
| **Nível de Energia** | 80% | 100% | Autonomia mínima para manobras orbitais. |
| **Pressão dos Tanques** | 3000 PSI | 4500 PSI | Pressão ideal para combustão eficiente. |
| **Status dos Módulos** | N/A | N/A | Deve ser rigorosamente **"Ativo"**. |

### 2.2 Justificativa Técnica
* **Margem de Erro:** Definimos o nível de energia em 80% (e não 50%) para aplicar o princípio da **Redundância**. No espaço, o que é "justo" é perigoso.
* **Ponto de Ruptura:** A pressão abaixo de 3000 PSI não fornece empuxo suficiente, enquanto acima de 4500 PSI arrisca a explosão das válvulas.

## 3. Algoritmo de Decisão (Pseudocódigo)
O pseudocódigo funciona como o projeto lógico da missão, descrevendo o passo a passo necessário para analisar os parâmetros de telemetria e determinar, com segurança, se a nave está pronta para decolar ou se a operação deve ser abortada

**ALGORITMO:** Missão Aurora

**ENTRADA:** Dataset 'telemetria_missao_aurora.csv'

**SAÍDA:** Status da Missão ("PRONTO PARA DECOLAR" ou "DECOLAGEM ABORTADA")



INÍCIO

    PARA cada registro no dataset FAÇA:
        SE (temp_interna < 18 OU temp_interna > 25) ENTÃO:
            RESULTADO = "ABORTAR: Falha Térmica Interna"
        
        SENÃO SE (temp_externa < -100 OU temp_externa > 60) ENTÃO:
            RESULTADO = "ABORTAR: Falha Térmica Externa"
        
        SENÃO SE (integridade_estrutural == 0) ENTÃO:
            RESULTADO = "ABORTAR: Falha na Integridade do Casco"
        
        SENÃO SE (nivel_energia < 80) ENTÃO:
            RESULTADO = "ABORTAR: Energia Insuficiente"
        
        SENÃO SE (pressao_tanques < 3000 OU pressao_tanques > 4500) ENTÃO:
            RESULTADO = "ABORTAR: Pressão fora da faixa"
        
        SENÃO SE (status_modulos != "Ativo") ENTÃO:
            RESULTADO = "ABORTAR: Módulos em estado crítico"
        
        SENÃO:
            RESULTADO = "PRONTO PARA DECOLAR (GO)"
        
        EXIBIR (Timestamp + RESULTADO)
    FIM PARA
FIM

## 4. Algoritmo em Python

Com a lógica validada pelo pseudocódigo, o script abaixo utiliza a biblioteca pandas para processar o dataset de telemetria. O objetivo é automatizar a verificação de cada registro, aplicando as regras de segurança aeroespacial para determinar o status operacional da Missão Aurora Siger

In [ ]:
import pandas as pd
import os

def analisar_decolagem(caminho_arquivo):
    """
    Lê a telemetria e aplica a lógica ("PRONTO PARA DECOLAR" ou "DECOLAGEM ABORTADA") baseada nos critérios de segurança.
    """
    # Verifica se o arquivo de dados existe
    if not os.path.exists(caminho_arquivo):
        print(f"Erro: O arquivo {caminho_arquivo} não foi encontrado.")
        return

    # Carregamento do dataset
    df = pd.read_csv(caminho_arquivo)
    resultados = []

    print(f"{'TIMESTAMP':<20} | {'ID':<4} | {'STATUS DA MISSÃO'}")
    print("-" * 70)

    # Iteração sobre os dados (Implementação da lógica do Pseudocódigo)
    for index, row in df.iterrows():
        ts = row['timestamp']
        id_leitura = int(row['leitura_id'])

        # Estrutura de decisão para os critérios de aceite
        if not (18 <= row['temp_interna'] <= 25):
            status = "DECOLAGEM ABORTADA: Falha Térmica Interna"
        elif not (-100 <= row['temp_externa'] <= 60):
            status = "DECOLAGEM ABORTADA: Falha Térmica Externa"
        elif row['integridade_estrutural'] == 0:
            status = "DECOLAGEM ABORTADA: Falha na Integridade"
        elif row['nivel_energia'] < 80:
            status = "DECOLAGEM ABORTADA: Energia Insuficiente"
        elif not (3000 <= row['pressao_tanques'] <= 4500):
            status = "DECOLAGEM ABORTADA: Pressão fora da faixa"
        elif row['status_modulos'] != 'Ativo':
            status = "DECOLAGEM ABORTADA: Módulos em Alerta/Crítico"
        else:
            status = "PRONTO PARA DECOLAR"

        # Saída de dados em tempo real
        print(f"{ts:<20} | {id_leitura:<4} | {status}")
        resultados.append(status)

    # Armazena os vereditos no DataFrame para futuras análises
    df['veredito_final'] = resultados

    # Salva o log da análise na pasta de dados
    df.to_csv('data/resultado_analise_missao.csv', index=False)
    return df

# Execução da análise
caminho_dados = 'data/telemetria_missao_aurora.csv'
df_resultado = analisar_decolagem(caminho_dados)

TIMESTAMP            | ID   | STATUS DA MISSÃO
----------------------------------------------------------------------
2026-04-01 02:01:00  | 1    | DECOLAGEM ABORTADA: Falha Térmica Externa
2026-04-01 02:01:01  | 2    | DECOLAGEM ABORTADA: Energia Insuficiente
2026-04-01 02:01:02  | 3    | DECOLAGEM ABORTADA: Falha Térmica Interna
2026-04-01 02:01:03  | 4    | DECOLAGEM ABORTADA: Falha Térmica Externa
2026-04-01 02:01:04  | 5    | DECOLAGEM ABORTADA: Falha Térmica Interna
2026-04-01 02:01:05  | 6    | PRONTO PARA DECOLAR
2026-04-01 02:01:06  | 7    | PRONTO PARA DECOLAR
2026-04-01 02:01:07  | 8    | DECOLAGEM ABORTADA: Falha na Integridade
2026-04-01 02:01:08  | 9    | DECOLAGEM ABORTADA: Módulos em Alerta/Crítico
2026-04-01 02:01:09  | 10   | PRONTO PARA DECOLAR
2026-04-01 02:01:10  | 11   | DECOLAGEM ABORTADA: Falha Térmica Interna
2026-04-01 02:01:11  | 12   | DECOLAGEM ABORTADA: Módulos em Alerta/Crítico
2026-04-01 02:01:12  | 13   | DECOLAGEM ABORTADA: Pressão fora da faixa
2026-0

## 5. Análise de Autonomia Energética

Nesta seção, calculamos se a energia disponível nos sistemas da Aurora Siger é suficiente para superar a fase de decolagem, considerando as perdas invevitáveis do processo.

### 5.1 Definição das métricas

Para o cálculo, utilizaremos os seguintes valores de referência:


**Capacidade Total ($C_{total}$):** $500\text{ kWh}

**Carga Atual ($L\%$):** Valor capturado pela telemetria (ex: $85\%$)

**Consumo na Decolagem ($E_{desc}$):** $60\text{ kWh}$ (estimado para os primeiros 10 minutos)

**Fator de Perda ($f_{perda}$):** $5\%$ ($0,05$) devido à resistência térmica e dissipação.


*italicized text*
---



### 5.2 Fórmulas Matemáticas Utilizadas

#### 1 Energia Disponível ($E_{disp}$):
Determina quanto de energia a nave realmente possui no momento:

$$E_{disp} = C_{total} \cdot \left(\frac{L\%}{100}\right)$$

#### 2 Energia Líquida após Perdas ($E_{liqd}$):
Aplica o fator de perda sobre a energia disponível:

$$E_{liqd} = E_{disp} \cdot (1 - f_{perda})$$


#### 3 Autonomia Pós-Decolagem ($E_{final}$):

Subtrai o consumo fixo da decolagem para verificar a reserva restante:

$$E_{final} = E_{liqd} - E_{desc}$$







## 5.3 Script de Cálculo de Autonomia

In [ ]:
def calcular_autonomia_missao(capacidade_total_kwh, carga_percentual, consumo_decolagem_kwh, fator_perda=0.05):
    """
    Realiza o cálculo de autonomia energética da Aurora Siger.
    """
    # 1. Energia Disponível
    energia_disponivel = capacidade_total_kwh * (carga_percentual / 100)

    # 2. Energia considerando perdas (ex: 5% de perda térmica)
    energia_liquida = energia_disponivel * (1 - fator_perda)

    # 3. Energia restante após o esforço da decolagem
    energia_final = energia_liquida - consumo_decolagem_kwh

    return {
        "disponivel": energia_disponivel,
        "liquida": energia_liquida,
        "final": energia_final,
        "status": "SUFICIENTE" if energia_final > 0 else "INSUFICIENTE"
    }

# Exemplo de uso com a primeira linha do seu DataFrame
# Vamos supor que a carga atual venha da coluna 'nivel_energia' do seu df_resultado
carga_teste = df_resultado.iloc[0]['nivel_energia']

resultado_energia = calcular_autonomia_missao(
    capacidade_total_kwh=500,
    carga_percentual=carga_teste,
    consumo_decolagem_kwh=60
)

print(f"--- RELATÓRIO ENERGÉTICO DA MISSÃO ---")
print(f"Carga da Bateria: {carga_teste}%")
print(f"Energia Disponível: {resultado_energia['disponivel']:.2f} kWh")
print(f"Energia (após 5% de perda): {resultado_energia['liquida']:.2f} kWh")
print(f"Reserva após Decolagem: {resultado_energia['final']:.2f} kWh")
print(f"Veredito Energético: {resultado_energia['status']}")

--- RELATÓRIO ENERGÉTICO DA MISSÃO ---
Carga da Bateria: 80.58%
Energia Disponível: 402.90 kWh
Energia (após 5% de perda): 382.75 kWh
Reserva após Decolagem: 322.75 kWh
Veredito Energético: SUFICIENTE


## 6. Análise Assistida por IA

Nesta seção, a inteligência artificial (LLM) analisou o dataset de 100 amostras capturadas. Abaixo, apresentamos o diagnóstico técnico baseado nos dados fornecidos.

### 6.1 Classificação dos Dados

Após o processamento das 100 leituras, a distribuição de status da missão foi a seguinte:

- **Decolagens Abortadas (66%):** A maioria dos registros apresentou violações de segurança. Diferente de falhas isoladas, os dados revelam um ambiente de teste com múltiplos estressores simultâneos.

- **Pronto para Decolar (34%):** Um total de 34 leituras foram validadas como seguras. Exemplos de conformidade total incluem os IDs 41, 42, 51, 71, 75, 76, 82 e 98.

- **Parâmetros mais críticos:** As maiores causas de interrupção foram a Pressão dos Tanques (23 falhas por estar fora da faixa de 3000-4500 PSI) e o Status dos Módulos (20 falhas por estados de Alerta ou Crítico).

### 6.2 Identificação de Anomalias Reais no Dataset

- **Instabilidade de Módulos:** Identificamos que o status dos módulos oscila para Alerta ou Critico de forma intermitente em diversos momentos (ex: IDs 9, 12, 14, 24, 35), sugerindo ruído eletrônico ou falha de comunicação nos barramentos de dados.

- **Falhas de Integridade Estrutural:** Ocorreram registros críticos de perda de integridade (valor 0) em pontos aleatórios, como nos IDs 5, 8, 23, 30 e 39, o que exige uma revisão imediata da estrutura física da aeronave.

- **Inconsistência de Pressão/Temperatura (ID 4):** Confirmamos que no ID 4, a temperatura externa atingiu -103.9°C, violando o limite de -100°C, enquanto a pressão interna permaneceu nominal (3446.48 PSI). Isso indica que o sistema de isolamento suportou o frio extremo, mas o protocolo ("PRONTO PARA DECOLAR/DECOLAGEM ABORTADA") agiu preventivamente.

### 6.3 Sugestões de Risco e Mitigação

- **Revisão do Sistema de Baterias:** O nível de energia atingiu o ponto mais baixo de 75.24% no ID 50, violando o limite mínimo de 80%. Mitigação: Implementar um sistema de redundância que impeça a queda abaixo de 85% durante a fase de checklist.

- **Substituição de Sensores de Casco:** Como a integridade estrutural oscila rapidamente (ex: ID 30 está em 0 e o ID 31 volta para 1), há indícios de mau contato nos sensores. Mitigação: Troca dos sensores piezoelétricos por modelos com maior tolerância a vibrações.

- **Ajuste de Range Térmico Externo:** O limite de -100°C foi excedido por margens mínimas, como no ID 1 (-100.75°C). Mitigação: Reavaliar se os componentes externos suportam até -105°C para evitar abortos desnecessários por variações de menos de 1 grau.

### 6.4 Prompt utilizado para esta análise

#### CONTEXTO
Atue como um Engenheiro Aeroespacial e Analista de Dados sênior da Missão Aurora Siger. Sua tarefa é auditar um dataset de telemetria de pré-decolagem e fornecer um relatório diagnóstico de segurança.

#### CRITÉRIOS DE SEGURANÇA (LIMITES TÉCNICOS)
Para que uma leitura seja considerada (Pronta para Decolar), ela deve atender a TODOS os critérios abaixo:
- Temperatura Interna: Entre 18°C e 25°C.
- Temperatura Externa: Entre -100°C e 60°C.
- Integridade Estrutural: Deve ser exatamente 1.
- Nível de Energia: Acima de 80%.
- Pressão dos Tanques: Entre 3000 e 4500 PSI.
- Status dos Módulos: Deve ser obrigatoriamente "Ativo".

#### TAREFAS DE ANÁLISE
Com base no CSV que fornecerei abaixo, execute:

- 1. CLASSIFICAÇÃO QUANTITATIVA:
   - Conte quantas das 100 leituras são "PRONTO PARA DECOLAR" e quantas são "ABORTAR".
   - Calcule a porcentagem exata de sucesso.
   - Identifique quais sensores (Pressão, Energia, Módulos, etc.) foram os maiores responsáveis pelos abortos.

- 2. IDENTIFICAÇÃO DE ANOMALIAS:
   - Identifique casos específicos de "quase sucesso" (ex: onde apenas um parâmetro falhou por pouco).
   - Analise temperatura Externa vs pressão e verifique a integridade.
   - Identifique instabilidades nos sensores de integridade estrutural (leituras que oscilam entre 0 e 1).

- 3. SUGESTÕES DE RISCO E MITIGAÇÃO:
   - Com base no valor mais baixo de energia encontrado e nas violações térmicas, sugira 3 melhorias de engenharia para a próxima janela de lançamento.

#### FORMATO DE SAÍDA
Forneça a análise dividida nos tópicos: "Classificação dos Dados", " Identificação de Anomalias Reais" e "Sugestões de Risco e Mitigação".



## 7. Reflexão Crítica

### 7.1 Ética e Responsabilidade

No desenvolvimento de algoritmos de missão crítica, a ética manifesta-se na integridade da informação. O código de verificação **("PRONT0 PARA DECOLAR" OU "DECOLAGEM ABORTADA")** que implementamos carrega a responsabilidade de preservar vidas humanas e investimentos bilionários. A negligência em uma faixa de segurança (como ignorar um sensor de pressão ou energia) não é apenas um erro de sintaxe, mas uma falha ética. A transparência na documentação e o rigor nos testes de telemetria são os compromissos fundamentais do desenvolvedor com a segurança da tripulação.

### 7.2 Impacto Social

Historicamente, a corrida espacial foi o motor de inovações que hoje salvam vidas na Terra. da purificação de água e avanços em telemedicina até o monitoramento climático via satélite. Ao analisarmos dados de uma missão como a Aurora, estamos aprimorando tecnologias de computação e eficiência energética que, eventualmente, serão aplicadas em cidades inteligentes e infraestruturas críticas usadas no planeta, democratizando o progresso científico

### 7.3 Sustentabilidade Tecnológica

A sustentabilidade na exploração moderna foca na redução de resíduos (detritos espaciais) e na eficiência de recursos. Em nosso projeto, a Análise Energética e a Análise Assistida por IA buscam otimizar o consumo de energia e prever falhas antes que elas ocorram, estendendo a vida útil dos equipamentos. Criar tecnologia sustentável significa projetar sistemas que não sejam descartáveis, mas que utilizem o máximo de seus recurso, garantindo que a exploração de novos mundos não comprometa o equilíbrio do nosso próprio planeta.
